In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.5 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
import glob
import shutil
from ultralytics import YOLO

# --- 1. Define Paths and Configuration ---
# Update this to the actual dataset path mounted in Kaggle
input_path = "/kaggle/input/datasets/gabrielfcarvalho/cardd-with-yolo-annotations-images-labels" 
output_path = "/kaggle/working/filtered_cardd"

splits = ['train', 'val', 'test']
target_classes = [0, 1]

# Counters for percentage calculation
class_counts = {0: 0, 1: 0}

# --- 2. Filter Dataset ---
print("Filtering dataset for classes 0 (dent) and 1 (scratch)...")
for split in splits:
    in_images_dir = os.path.join(input_path, split, 'images')
    in_labels_dir = os.path.join(input_path, split, 'labels')
    
    out_images_dir = os.path.join(output_path, split, 'images')
    out_labels_dir = os.path.join(output_path, split, 'labels')
    
    # Create new directories in /kaggle/working/
    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_labels_dir, exist_ok=True)
    
    if not os.path.exists(in_labels_dir):
        continue
        
    for label_name in os.listdir(in_labels_dir):
        if not label_name.endswith('.txt'):
            continue
            
        label_file = os.path.join(in_labels_dir, label_name)
        with open(label_file, 'r') as f:
            lines = f.readlines()
            
        kept_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
                
            class_id = int(parts[0])
            if class_id in target_classes:
                kept_lines.append(line)
                class_counts[class_id] += 1
                
        # If the file contains our target classes, save it and copy the image
        if kept_lines:
            with open(os.path.join(out_labels_dir, label_name), 'w') as f:
                f.writelines(kept_lines)
                
            # Dynamically locate and copy the corresponding image
            img_base = os.path.splitext(label_name)[0]
            img_files = glob.glob(os.path.join(in_images_dir, f"{img_base}.*"))
            if img_files:
                shutil.copy(img_files[0], os.path.join(out_images_dir, os.path.basename(img_files[0])))

# --- 3. Calculate Percentages ---
print("\n--- Class Distribution in Filtered Data ---")
total_annotations = sum(class_counts.values())

if total_annotations > 0:
    for cls_id in target_classes:
        percentage = (class_counts[cls_id] / total_annotations) * 100
        print(f"Class {cls_id}: {percentage:.2f}% ({class_counts[cls_id]} instances)")
else:
    print("No annotations found for the target classes. Check your dataset paths.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Filtering dataset for classes 0 (dent) and 1 (scratch)...

--- Class Distribution in Filtered Data ---
Class 0: 41.43% (2543 instances)
Class 1: 58.57% (3595 instances)


In [5]:

# --- 4. Generate data.yaml ---
yaml_content = f"""
train: {os.path.join(output_path, 'train', 'images')}
val: {os.path.join(output_path, 'val', 'images')}
test: {os.path.join(output_path, 'test', 'images')}

nc: 2
names: ['dent', 'scratch']
"""

yaml_path = os.path.join(output_path, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())
print(f"\nGenerated dataset configuration at: {yaml_path}")

# --- 5. Train the YOLO Model ---
print("\n--- Starting YOLO Training ---")
# Initializing a pre-trained YOLOv8 nano model
model = YOLO('yolo26n.pt') 
# model = YOLO('/kaggle/working/yolo_training/dent_scratch_detector/weights/best.pt') 


# Train the model on the newly created filtered dataset
results = model.train(
    data=yaml_path,
    epochs=50,      # Adjust based on your Kaggle GPU time
    imgsz=1280,
    batch=16,
    project='/kaggle/working/yolo26',
    name='dent_scratch_detector',
    patience = 10
)


Generated dataset configuration at: /kaggle/working/filtered_cardd/data.yaml

--- Starting YOLO Training ---
Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/filtered_cardd/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, m

In [6]:
import os
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# 1. Load the trained model
# This path points to the 'best.pt' weights generated from the previous training block
model_path = '/kaggle/working/yolo26/dent_scratch_detector/weights/best.pt'
model = YOLO(model_path)

# 2. Evaluate the model on the validation dataset
# The .val() method automatically computes metrics and plots evaluation graphs
print("Evaluating the model...")
metrics = model.val(project='/kaggle/working/yolo26', name='val_results')

# 3. Extract and Calculate Metrics
precision = metrics.results_dict['metrics/precision(B)']
recall = metrics.results_dict['metrics/recall(B)']

# Calculate the F1 Score
if (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
else:
    f1_score = 0.0

print("\n--- Evaluation Metrics ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1_score:.4f}")

# 4. Display the Confusion Matrix
# Ultralytics saves the matrix as a .png file in the output directory
conf_matrix_path = '/kaggle/working/yolo_training/val_results/confusion_matrix.png'

if os.path.exists(conf_matrix_path):
    print(f"\nDisplaying Confusion Matrix from: {conf_matrix_path}")
    
    # Load and plot the image using matplotlib
    img = Image.open(conf_matrix_path)
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')  # Hide the axes for a cleaner view
    plt.show()
    
else:
    print(f"Confusion matrix image not found at {conf_matrix_path}. Check your directory structure.")

Evaluating the model...
Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4547.9±879.2 MB/s, size: 650.0 KB)
val: Scanning /kaggle/working/filtered_cardd/val/labels.cache... 603 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 603/603 281.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 3.0it/s 12.5s0.3s
                   all        603       1229      0.612      0.524      0.529      0.287
                  dent        352        501      0.628      0.511       0.52      0.273
               scratch        431        728      0.595      0.538      0.538      0.301
Speed: 3.9ms preprocess, 12.3ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /kaggle/working/yolo26/val_results

--- Evaluation Metrics ---
Precision: 0.6118